# Error Analysis — DistilBERT on IMDB Test Set

**Course:** Statistical ML / Data Science Capstone  
**Purpose:** Qualitative + quantitative analysis of **false positives (FP)** and **false negatives (FN)**  
**Source:** `artifacts/results/error_analysis_test.csv` (from `make error-analysis`)  
**Use in defense:** 2–3 concrete misclassification examples + error-type breakdown  

---

In [ ]:
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
GOLD = '#e8c547'
POS_COLOR = '#2ecc71'
NEG_COLOR = '#e74c3c'
FP_COLOR = '#e67e22'
FN_COLOR = '#9b59b6'

ROOT = Path('..').resolve()
RESULTS = ROOT / 'artifacts' / 'results'
ERROR_CSV = RESULTS / 'error_analysis_test.csv'
RESULTS.mkdir(parents=True, exist_ok=True)

if not ERROR_CSV.is_file():
    raise FileNotFoundError(
        f'Missing {ERROR_CSV}. Run from project root: make error-analysis'
    )

df = pd.read_csv(ERROR_CSV)
print(f'Loaded {len(df):,} misclassified test reviews')
print(df.columns.tolist())
df.head(2)

In [ ]:
# ── Error type counts ─────────────────────────────────────────────
counts = df['error_type'].value_counts()
fp_n = int(counts.get('false_positive', 0))
fn_n = int(counts.get('false_negative', 0))
total_err = fp_n + fn_n

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [FP_COLOR if k == 'false_positive' else FN_COLOR for k in counts.index]
bars = axes[0].bar(counts.index.str.replace('_', ' ').str.title(), counts.values,
                   color=colors, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{v:,}', ha='center', color='white', fontsize=12)
axes[0].set_title('Misclassifications by Type', fontsize=14, color=GOLD)
axes[0].set_ylabel('Count')

axes[1].pie([fp_n, fn_n], labels=['False Positive', 'False Negative'],
            colors=[FP_COLOR, FN_COLOR], autopct='%1.1f%%', startangle=90,
            textprops={'color': 'white'})
axes[1].set_title('FP vs FN Share', fontsize=14, color=GOLD)

fig.suptitle(f'Test-Set Errors (n = {total_err:,})', fontsize=15, color=GOLD, y=1.02)
plt.tight_layout()
fig.savefig(RESULTS / 'error_type_counts.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'FP rate among errors: {fp_n/total_err*100:.1f}%  |  FN: {fn_n/total_err*100:.1f}%')

In [ ]:
# ── Confidence distribution (P(positive)) by error type ───────────
fig, ax = plt.subplots(figsize=(12, 5))
for etype, color in [('false_positive', FP_COLOR), ('false_negative', FN_COLOR)]:
    sub = df.loc[df['error_type'] == etype, 'prob_positive']
    ax.hist(sub, bins=40, alpha=0.65, color=color, label=etype.replace('_', ' ').title(),
            edgecolor='white', linewidth=0.3)
ax.axvline(0.5, color=GOLD, linestyle='--', linewidth=2, label='Threshold τ = 0.5')
ax.set_xlabel('Predicted P(Fresh / positive)')
ax.set_ylabel('Frequency')
ax.set_title('Model Confidence on Misclassified Reviews', fontsize=14, color=GOLD)
ax.legend()
plt.tight_layout()
fig.savefig(RESULTS / 'error_confidence_dist.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

print('FP — mean P(pos):', df.loc[df.error_type=='false_positive', 'prob_positive'].mean().round(3))
print('FN — mean P(pos):', df.loc[df.error_type=='false_negative', 'prob_positive'].mean().round(3))

In [ ]:
# ── Review length of errors ───────────────────────────────────────
df['preview_len'] = df['text_preview'].astype(str).str.len()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='error_type', y='preview_len',
            order=['false_positive', 'false_negative'],
            palette=[FP_COLOR, FN_COLOR], ax=ax)
ax.set_xticklabels(['False Positive', 'False Negative'])
ax.set_title('Preview Length by Error Type', fontsize=14, color=GOLD)
ax.set_xlabel('')
ax.set_ylabel('Characters (preview field)')
plt.tight_layout()
plt.show()

In [ ]:
def show_examples(error_type: str, n: int = 3, sort_by: str = 'prob_positive'):
  sub = df[df['error_type'] == error_type].copy()
  if sort_by == 'prob_positive':
    sub = sub.sort_values('prob_positive', ascending=(error_type == 'false_negative'))
  else:
    sub = sub.sort_values('prob_positive', ascending=False)
  title = 'FALSE POSITIVE' if error_type == 'false_positive' else 'FALSE NEGATIVE'
  true_lbl = 'Rotten (0)' if error_type == 'false_positive' else 'Fresh (1)'
  pred_lbl = 'Fresh (1)' if error_type == 'false_positive' else 'Rotten (0)'
  print('=' * 88)
  print(f'{title} — top {n} examples for defense slides')
  print('=' * 88)
  for i, row in sub.head(n).iterrows():
    print(f"\n[{title}]  True={true_lbl}  Pred={pred_lbl}  P(pos)={row['prob_positive']:.3f}")
    wrapped = textwrap.fill(str(row['text_preview']), width=84, initial_indent='  ', subsequent_indent='  ')
    print(wrapped)

show_examples('false_positive', n=3)
show_examples('false_negative', n=3)

In [ ]:
# ── Export defense table (CSV) ────────────────────────────────────
defense = df.groupby('error_type').head(5)[
    ['error_type', 'true_label', 'predicted_label', 'prob_positive', 'text_preview']
].copy()
out = RESULTS / 'error_defense_samples.csv'
defense.to_csv(out, index=False)
print(f'Wrote {out}')
defense.style.set_caption('Top-5 FP/FN samples for báo cáo')

---

## Summary for báo cáo / defense

| Finding | Interpretation |
|---------|----------------|
| **FP (predicted Fresh, actually Rotten)** | Often mixed reviews, niche docs, or atypical praise — model overweights positive cues. |
| **FN (predicted Rotten, actually Fresh)** | Reviews with criticism + praise, sarcasm, or genre-specific language (e.g. action franchises). |
| **Confidence** | Many errors sit **near τ = 0.5** → threshold tuning or calibration may help; not all errors are high-confidence mistakes. |
| **Length** | Long reviews truncated at 256 tokens may lose decisive sentiment cues (links to EDA notebook). |

### Talking points (30 s each)

1. *"We export every test-set error with type, probability, and text preview for auditability."*  
2. *"FP/FN examples show the model struggles with mixed sentiment — expected for bag-of-words and challenging for transformers too."*  
3. *"These cases motivate future work: aspect-level sentiment and human evaluation."*

**Figures saved:** `error_type_counts.png`, `error_confidence_dist.png`  
**Next:** `03_model_comparison.ipynb` — statistical comparison vs baselines  

---